# Actor-Critic SQL Generation — Demo Notebook

This notebook demonstrates the dual-agent SQL generation workflow:

| Agent | Model | Role |
|-------|-------|------|
| **Actor** | Anthropic Claude (via Vertex AI Model Garden) | Generates SQL from natural language |
| **Critic** | Google Gemini (via Vertex AI) | Validates and corrects the SQL |

The workflow is orchestrated by **LangGraph** with automatic tracing via **LangSmith**.

**Prerequisites:**
- Running on Vertex AI Workbench (or any environment with GCP credentials)
- Access to Vertex AI Model Garden (Claude) and Gemini API
- A LangSmith account (optional, for tracing)

---

## 1. Install Dependencies

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Choose **one** of the three methods below to configure the workflow.
On Vertex AI Workbench, GCP authentication is automatic via the attached service account.

---

### Method A — `.env` file (recommended for local development)

Copy `.env.example` to `.env` and fill in your values, then run the cell below.

In [ ]:
from workflow.config import WorkflowConfig

# Reads from .env in the sql_generation/ directory
config = WorkflowConfig.from_env()

### Method B — Hardcoded values (quick notebook experimentation)

Replace the placeholder values and run this cell **instead of** Method A.

In [ ]:
from workflow.config import WorkflowConfig

config = WorkflowConfig.from_values(
    gcp_project_id="your-gcp-project-id",      # ← replace
    gcp_location_gemini="us-central1",
    gcp_location_claude="us-east5",
    actor_model="claude-sonnet-4-20250514",
    critic_model="gemini-2.5-flash",
    max_attempts=3,
    use_case="tpch",
    langsmith_api_key="your-langsmith-api-key",  # ← replace (or "" to disable)
    langsmith_project="sql-generation-actor-critic",
)

### Method C — Google Cloud Secret Manager (production)

Secrets are stored in Secret Manager with a common prefix. See `README.md` for setup.

In [ ]:
from workflow.config import WorkflowConfig

config = WorkflowConfig.from_secret_manager(
    gcp_project_id="your-gcp-project-id",  # ← replace
    secret_prefix="sql-gen",
)

---

## 3. Build the Workflow

The `build_sql_workflow` function initializes both LLMs, reads the prompt
templates and grounding documents, and compiles the LangGraph state machine.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(name)-35s │ %(levelname)-8s │ %(message)s",
    datefmt="%H:%M:%S",
)

from workflow.graph import build_sql_workflow

graph = build_sql_workflow(config)
print("Workflow compiled. Nodes:", list(graph.get_graph().nodes.keys()))

### Visualize the graph (optional)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"Graph visualization not available: {exc}")
    print("Install graphviz or use LangSmith to view the graph.")

---

## 4. Run Example Queries

Each invocation sends a natural language question through the full
Actor → Critic → (correction loop) → Finalize pipeline.

If LangSmith is configured, every node execution and LLM call is traced automatically.

### Example 1 — Simple aggregation

A straightforward top-N query exercising `GROUP BY`, `ORDER BY`, and `LIMIT`.

In [ ]:
result_1 = graph.invoke({
    "user_query": (
        "Show me the top 10 nations by total revenue. "
        "Revenue is defined as l_extendedprice * (1 - l_discount)."
    )
})

print(f"Status : {result_1['status']}")
print(f"Attempts: {result_1['attempt']}")
print(f"Verdict : {result_1.get('critic_verdict', 'N/A')}")
print("\n── Generated SQL ─────────────────────────────────────")
print(result_1["final_sql"])
print("\n── Explanation ───────────────────────────────────────")
print(result_1["final_explanation"])

### Example 2 — Complex CTE with window functions

Requires CTEs, window functions (`RANK`, `SUM OVER`), and multi-table joins.

In [ ]:
result_2 = graph.invoke({
    "user_query": (
        "For each region, rank nations by their total revenue and show "
        "each nation's percentage contribution to the regional total. "
        "Include only the top 3 nations per region. "
        "Use window functions for ranking and percentage calculation."
    )
})

print(f"Status : {result_2['status']}")
print(f"Attempts: {result_2['attempt']}")
print(f"Verdict : {result_2.get('critic_verdict', 'N/A')}")
print("\n── Generated SQL ─────────────────────────────────────")
print(result_2["final_sql"])
print("\n── Explanation ───────────────────────────────────────")
print(result_2["final_explanation"])

### Example 3 — Running totals and self-join

Exercises cumulative aggregation and date-range filtering.

In [ ]:
result_3 = graph.invoke({
    "user_query": (
        "Show the month-over-month running total of revenue for 1995, "
        "broken down by order priority. For each month, include the "
        "monthly revenue, the cumulative running total up to that month, "
        "and the month-over-month percentage change."
    )
})

print(f"Status : {result_3['status']}")
print(f"Attempts: {result_3['attempt']}")
print(f"Verdict : {result_3.get('critic_verdict', 'N/A')}")
print("\n── Generated SQL ─────────────────────────────────────")
print(result_3["final_sql"])
print("\n── Explanation ───────────────────────────────────────")
print(result_3["final_explanation"])

---

## 5. Inspect Correction History

When the Critic rejects or corrects the Actor's SQL, each attempt is
recorded in `correction_history`. This lets you trace the refinement
process without leaving the notebook.

In [ ]:
import json

for i, result in enumerate([result_1, result_2, result_3], 1):
    history = result.get("correction_history", [])
    print(f"\n{'='*60}")
    print(f"Example {i}: {result['status'].upper()} after {result['attempt']} attempt(s)")
    print(f"{'='*60}")
    for entry in history:
        print(f"  Attempt {entry['attempt']} ({entry['source']}):")
        sql_preview = entry['sql'][:120].replace('\n', ' ')
        print(f"    SQL: {sql_preview}..." if len(entry['sql']) > 120 else f"    SQL: {sql_preview}")
    if result.get("critic_issues"):
        print(f"  Last critic issues:")
        for issue in result["critic_issues"]:
            print(f"    [{issue['severity'].upper()}] {issue['category']}: {issue['description']}")

---

## 6. LangSmith Tracing

If you configured a `LANGSMITH_API_KEY`, every graph invocation above has
been automatically traced. Each trace contains:

- **Top-level run**: The full `graph.invoke()` call
  - **assemble_context**: Prompt assembly (no LLM call)
  - **generate_sql**: Claude invocation + prompt + response
  - **validate_sql**: Gemini invocation + prompt + response
  - **apply_correction** (if triggered): State update
  - **finalize**: Terminal state

### View traces in the LangSmith UI

1. Open [smith.langchain.com](https://smith.langchain.com)
2. Navigate to the project configured in `LANGSMITH_PROJECT`
3. Click on any run to see the full node-by-node execution, LLM inputs/outputs, and latencies

### Programmatic access

In [ ]:
import os

if os.environ.get("LANGSMITH_TRACING") == "true":
    from langsmith import Client

    ls_client = Client()
    project = os.environ.get("LANGSMITH_PROJECT", "sql-generation-actor-critic")

    runs = list(ls_client.list_runs(
        project_name=project,
        limit=5,
    ))

    print(f"Recent runs in project '{project}':")
    for run in runs:
        print(
            f"  {run.name:30s}  status={run.status:12s}  "
            f"latency={run.total_tokens or 'N/A'}  "
            f"url=https://smith.langchain.com/o/default/projects/p/{project}/r/{run.id}"
        )
else:
    print("LangSmith tracing is not enabled. Set LANGSMITH_API_KEY to enable.")

---

## 7. Custom Query

Try your own natural language question against the TPC-H schema.

In [ ]:
custom_query = """Which suppliers in Europe have the highest number of 
parts available with a supply cost below the average supply cost 
for their nation? Show the top 5 suppliers with their nation, 
part count, and average supply cost."""

result_custom = graph.invoke({"user_query": custom_query})

print(f"Status: {result_custom['status']} | Attempts: {result_custom['attempt']}")
print(f"\n── SQL ───────────────────────────────────────────────")
print(result_custom["final_sql"])
print(f"\n── Explanation ───────────────────────────────────────")
print(result_custom["final_explanation"])
if result_custom.get("critic_issues"):
    print(f"\n── Issues ────────────────────────────────────────────")
    for issue in result_custom["critic_issues"]:
        print(f"  [{issue['severity']}] {issue['category']}: {issue['description']}")